# COSC3047 Assignment 2: Social Media and Network Analysis

## Diffusion of Trump's 2025 Tariff Discourse Across Australian Reddit

This notebook contains the full analysis pipeline used for the Assignment 2 report, including Reddit data collection, reply-network construction, centrality analysis, Louvain community detection, cross-subreddit bridge analysis, VADER sentiment analysis, LDA topic/frame modelling, temporal diffusion analysis, and exploratory stance validation.

## Archived Post Collection Code

This cell preserves the original Arctic Shift API code used to collect tariff-related Reddit posts. It is kept for transparency but should not be rerun during marking because the final notebook uses cached CSV files from the `data/` folder.

In [ ]:
'''
import requests
import pandas as pd
import time
import os
from datetime import datetime, timezone

HEADERS = {
    'User-Agent': 'RMIT-COSC2671-Assignment2 (research; contact via Canvas)'
}

BASE = "https://arctic-shift.photon-reddit.com"

# helper: paginate over arctic-shift search endpoint
def arctic_search(endpoint, params, page_size=100, max_pages=200, sleep=1.0):
    """
    Paginate backwards in time using before.
    endpoint is e.g. '/api/posts/search' or '/api/comments/search'.
    params should include at minimum: subreddit, after, before, and (optionally) query.
    """
    url = BASE + endpoint
    results = []
    p = dict(params)
    p['limit'] = page_size
    p['sort'] = 'desc'  # newest first; we'll walk backwards using before

    after_ts = int(p['after'])

    for _ in range(max_pages):
        r = requests.get(url, headers=HEADERS, params=p, timeout=60)

        if r.status_code != 200:
            # Show what the server actually said — usually a clear param error
            print(f"  HTTP {r.status_code}: {r.text[:300]}")
            break

        batch = r.json().get('data', [])
        if not batch:
            break

        results.extend(batch)

        oldest = min(item['created_utc'] for item in batch)
        if len(batch) < page_size or oldest <= after_ts:
            break

        # Step the window back: next page ends at the oldest item of this page
        p['before'] = oldest
        time.sleep(sleep)

    return results

# step 1: collect posts
# Trump's "Liberation Day" tariffs were April 2, 2025. Capture a wide window:
# from before the announcement (to see baseline) through the months after.
AFTER  = int(datetime(2025, 1, 1,  tzinfo=timezone.utc).timestamp())
BEFORE = int(datetime(2025, 12, 31, tzinfo=timezone.utc).timestamp())

subreddits = ['AusFinance', 'ASX_Bets', 'australia', 'AusEcon', 'ASX', 'AustralianPolitics']

queries = [
    'trump tariff',
    'liberation day',
    'trade war',
    'tariff',          # broad — narrow later in pandas if needed
    'trump trade',
    'china tariff',
]

all_posts = []
for sub in subreddits:
    for q in queries:
        print(f"Posts: r/{sub}  query='{q}'")
        posts = arctic_search(
            '/api/posts/search',
            params={
                'subreddit': sub,
                'query': q,            # <-- the fix: was 'q', should be 'query'
                'after': AFTER,
                'before': BEFORE,
            },
        )
        print(f"  → {len(posts)} posts")
        for p in posts:
            all_posts.append({
                'id':            p.get('id'),
                'title':         p.get('title', '') or '',
                'selftext':      p.get('selftext', '') or '',
                'author':        p.get('author', '') or '',
                'score':         p.get('score', 0),
                'upvote_ratio':  p.get('upvote_ratio'),
                'num_comments':  p.get('num_comments', 0),
                'created_utc':   p.get('created_utc'),
                'subreddit':     p.get('subreddit', '') or '',
                'permalink':     p.get('permalink', '') or '',
                'matched_query': q,
            })
        time.sleep(1)

# safe DataFrame build (no KeyError if we collected nothing)
posts_df = pd.DataFrame(all_posts)
if len(posts_df) == 0:
    raise RuntimeError(
        "No posts collected. Inspect the HTTP error messages above before continuing."
    )

posts_df = posts_df.drop_duplicates(subset='id').reset_index(drop=True)
posts_df['date'] = pd.to_datetime(posts_df['created_utc'], unit='s', utc=True)

os.makedirs('data', exist_ok=True)
posts_df.to_csv('data/reddit_posts.csv', index=False)

print(f"\n--- POSTS SUMMARY ---")
print(f"Total unique posts: {len(posts_df)}")
print(f"\nPer subreddit:")
print(posts_df['subreddit'].value_counts())
print(f"\nBy month:")
print(posts_df.groupby(posts_df['date'].dt.to_period('M')).size())
'''

## Data Loading and Preparation

Loading the cached Reddit post and comment datasets from the `data/` folder. These cached files are used to make the notebook reproducible without rerunning long API collection steps.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle
import os

posts_df = pd.read_csv("data/reddit_posts.csv")
comments_df = pd.read_csv("data/reddit_comments.csv")

posts_df["date"] = pd.to_datetime(posts_df["created_utc"], unit="s", utc=True)
comments_df["date"] = pd.to_datetime(comments_df["created_utc"], unit="s", utc=True)

print(f"Loaded {len(posts_df)} posts")
print(f"Loaded {len(comments_df)} comments")

## Archived Comment Collection Code

This cell preserves the original code used to collect comments for each Reddit post. It is kept for transparency but should not be rerun because the cleaned comment dataset is already saved in `data/reddit_comments.csv`.

In [ ]:
'''
# step 2: collect comments for each post (needed for network analysis)
# We'll cap how many comments per post to keep collection bounded.
# The /api/comments/search endpoint with link_id pulls all comments under one post.

def fetch_comments_for_post(post_id, limit_per_call=100):
    """Pull all comments under a post via search, paginating with `before`."""
    return arctic_search(
        '/api/comments/search',
        params={
            'link_id': post_id,
            'after':  AFTER,
            'before': BEFORE,
        },
        page_size=limit_per_call,
        sleep=0.5,
    )

# we prioritise posts with actual discussion, no point pulling comments on a 0-comment post
posts_with_discussion = (
    posts_df[posts_df['num_comments'] >= 3]
    .sort_values('num_comments', ascending=False)
    .head(500)   # tune this — start small, expand once code works end-to-end
)

print(f"Fetching comments for {len(posts_with_discussion)} posts...")

all_comments = []
for i, row in enumerate(posts_with_discussion.itertuples(), 1):
    if i % 25 == 0:
        print(f"  [{i}/{len(posts_with_discussion)}] {len(all_comments)} comments so far")
    try:
        comments = fetch_comments_for_post(row.id)
    except Exception as e:
        print(f"  failed on {row.id}: {e}")
        continue

    for c in comments:
        all_comments.append({
            'id':          c.get('id'),
            'link_id':     c.get('link_id', '').replace('t3_', ''),  # parent post id
            'parent_id':   c.get('parent_id', ''),                   # t1_xxx (comment) or t3_xxx (post)
            'author':      c.get('author', '') or '',
            'body':        c.get('body', '') or '',
            'score':       c.get('score', 0),
            'created_utc': c.get('created_utc'),
            'subreddit':   c.get('subreddit', '') or '',
        })

comments_df = pd.DataFrame(all_comments).drop_duplicates(subset='id').reset_index(drop=True)
comments_df['date'] = pd.to_datetime(comments_df['created_utc'], unit='s', utc=True)
comments_df = comments_df[~comments_df['author'].isin(['[deleted]', 'AutoModerator', ''])]

comments_df.to_csv('data/reddit_comments.csv', index=False)

print(f"\n--- COMMENTS SUMMARY ---")
print(f"Total unique comments: {len(comments_df)}")
print(f"Unique commenters: {comments_df['author'].nunique()}")
print(f"Per subreddit:")
print(comments_df['subreddit'].value_counts())

print("This cell is commented, it is not needed but still here as progress history or incase something happens to the data")
'''

## Directed Reply Network Construction

This section builds the directed weighted reply network from Reddit comment-parent relationships. Users are represented as nodes, and replies are represented as directed edges from the replying user to the parent author.

In [ ]:
from collections import Counter

posts_df    = pd.read_csv('data/reddit_posts.csv')
comments_df = pd.read_csv('data/reddit_comments.csv')

# some authors come through as NaN/[deleted]/AutoModerator, we exclude them consistently
EXCLUDE = {'[deleted]', '[removed]', 'AutoModerator', '', None}

posts_df    = posts_df[~posts_df['author'].isin(EXCLUDE)].dropna(subset=['author'])
comments_df = comments_df[~comments_df['author'].isin(EXCLUDE)].dropna(subset=['author'])

# comment_id -> author
comment_author = dict(zip(comments_df['id'], comments_df['author']))
# post_id -> author
post_author    = dict(zip(posts_df['id'], posts_df['author']))

# build directed reply edges: replier -> parent author
edges = []          # (src, dst, subreddit, post_id, created_utc)
unresolved = 0      # parent author not in our dataset

for c in comments_df.itertuples():
    src = c.author
    parent = c.parent_id
    if not isinstance(parent, str) or '_' not in parent:
        continue

    kind, pid = parent.split('_', 1)
    if kind == 't1':
        dst = comment_author.get(pid)
    elif kind == 't3':
        dst = post_author.get(pid)
    else:
        dst = None

    if dst is None or dst in EXCLUDE:
        unresolved += 1
        continue
    if src == dst: # self-replies aren't informative
        continue

    edges.append((src, dst, c.subreddit, c.link_id, c.created_utc))

edges_df = pd.DataFrame(edges, columns=['src', 'dst', 'subreddit', 'post_id', 'created_utc'])
print(f"Total directed interactions: {len(edges_df):,}")
print(f"Unresolved (parent outside our dataset): {unresolved:,}")
print(f"Unique users in network: {len(set(edges_df['src']) | set(edges_df['dst'])):,}")

# build weighted directed graph
weighted = (edges_df
            .groupby(['src', 'dst'])
            .size()
            .reset_index(name='weight'))

G = nx.from_pandas_edgelist(
    weighted, source='src', target='dst',
    edge_attr='weight', create_using=nx.DiGraph
)
print(f"\nGraph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Density: {nx.density(G):.5f}")

with open('data/reply_graph.pkl', 'wb') as f:
    pickle.dump(G, f)
edges_df.to_csv('data/edges.csv', index=False)

## Centrality Analysis: Influential User Roles

This section computes multiple centrality measures to identify different types of influential users, including attention-attracting users, active broadcasters, recursively important users, and bridge users.

In [ ]:
import pandas as pd
import networkx as nx
import pickle

# centrality section — different measures capture different "influence"

# if G is not already in memory, load the saved reply graph
try:
    G
except NameError:
    with open("data/reply_graph.pkl", "rb") as f:
        G = pickle.load(f)

# focus on largest weakly connected component for meaningful centrality measures
largest_wcc = max(nx.weakly_connected_components(G), key=len)
Gc = G.subgraph(largest_wcc).copy()

print(
    f"Largest WCC: {Gc.number_of_nodes():,} nodes, {Gc.number_of_edges():,} edges "
    f"({Gc.number_of_nodes() / G.number_of_nodes():.1%} of full graph)"
)

print("Computing centralities...")

# weighted in-degree: how often this user is replied to
in_deg = dict(Gc.in_degree(weight="weight"))

# weighted out-degree: how often this user replies to others
out_deg = dict(Gc.out_degree(weight="weight"))

# PageRank: recursive influence based on reply structure
pagerank = nx.pagerank(Gc, weight="weight")

# betweenness: bridge role across the network
# estimated using k=500 for reproducibility and practical runtime
betweenness = nx.betweenness_centrality(
    Gc,
    k=min(500, Gc.number_of_nodes()),
    weight="weight",
    seed=42
)

cent_df = pd.DataFrame({
    "user": list(Gc.nodes()),
    "in_degree": [in_deg.get(u, 0) for u in Gc.nodes()],
    "out_degree": [out_deg.get(u, 0) for u in Gc.nodes()],
    "pagerank": [pagerank.get(u, 0) for u in Gc.nodes()],
    "betweenness": [betweenness.get(u, 0) for u in Gc.nodes()],
})

cent_df.to_csv("data/centrality.csv", index=False)

# who matters? look at top 15 by each measure
for col in ["in_degree", "out_degree", "pagerank", "betweenness"]:
    print(f"\nTop 15 by {col}:")
    print(cent_df.nlargest(15, col)[["user", col]].to_string(index=False))

## Louvain Community Detection

This section applies Louvain community detection to identify user clusters in the reply network and assess whether communities align more strongly with subreddit boundaries or ideological positions.

In [ ]:

# community detection section — Louvain on the undirected version

# Louvain needs an undirected graph; we collapse reciprocal edges
# by adding weights (A->B weight 3, B->A weight 2 becomes A-B weight 5)
G_und = Gc.to_undirected()
for u, v, d in G_und.edges(data=True):
    # in a digraph -> undirected conversion, weights from both directions need merging
    pass  # nx.to_undirected handles this correctly if we sum in a second pass

# cleaner: rebuild undirected with added weights
und_weighted = (edges_df
                .assign(pair=edges_df.apply(
                    lambda r: tuple(sorted([r['src'], r['dst']])), axis=1))
                .groupby('pair').size().reset_index(name='weight'))
und_weighted[['u', 'v']] = pd.DataFrame(und_weighted['pair'].tolist(), index=und_weighted.index)
G_und = nx.from_pandas_edgelist(und_weighted, source='u', target='v',
                                edge_attr='weight', create_using=nx.Graph)
G_und = G_und.subgraph(largest_wcc).copy()

communities = nx.community.louvain_communities(G_und, weight='weight', seed=42, resolution=1.0)
communities = sorted(communities, key=len, reverse=True)

print(f"\nFound {len(communities)} communities")
print(f"Sizes of top 10: {[len(c) for c in communities[:10]]}")

# tag each user with their community id
user_to_comm = {}
for cid, members in enumerate(communities):
    for u in members:
        user_to_comm[u] = cid

cent_df['community'] = cent_df['user'].map(user_to_comm)
cent_df.to_csv('data/centrality.csv', index=False)

# how big are the major communities analysis
print("\nTop users in each of the 5 largest communities:")
for cid in range(5):
    top = cent_df[cent_df['community'] == cid].nlargest(5, 'pagerank')
    print(f"\n--- Community {cid} ({len(communities[cid])} users) ---")
    print(top[['user', 'pagerank', 'in_degree']].to_string(index=False))

## Community Composition by Subreddit

This section checks whether Louvain-detected communities represent ideological groups or whether they mainly reflect subreddit boundaries.

In [ ]:
# are the communities just subreddit echo chambers? for each community, what's the subreddit distribution of activity?
user_main_sub = (comments_df.groupby('author')['subreddit']
                 .agg(lambda x: x.value_counts().index[0])
                 .to_dict())

cent_df['main_subreddit'] = cent_df['user'].map(user_main_sub)

print("Top-5 communities — subreddit makeup of their members:")
for cid in range(5):
    members = cent_df[cent_df['community'] == cid]
    sub_dist = members['main_subreddit'].value_counts(normalize=True).head(3)
    print(f"\nCommunity {cid} ({len(members)} users):")
    for sub, pct in sub_dist.items():
        print(f"  {sub:25s} {pct:.1%}")

cent_df.to_csv('data/centrality.csv', index=False)

## Final Louvain Composition Figure Export

This cell regenerates the Louvain subreddit composition figure from the final validated centrality file. It uses the 32-community result reported in the final report and saves the correct figure to the `figures/` folder.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

cent_df = pd.read_csv("data/centrality.csv")

print("Communities:", cent_df["community"].nunique())

top_communities = cent_df["community"].value_counts().head(10).index.tolist()

plot_df = (
    cent_df[cent_df["community"].isin(top_communities)]
    .groupby(["community", "main_subreddit"])
    .size()
    .reset_index(name="users")
)

plot_df["share"] = plot_df.groupby("community")["users"].transform(lambda x: x / x.sum() * 100)

pivot = plot_df.pivot_table(
    index="community",
    columns="main_subreddit",
    values="share",
    fill_value=0
)

pivot = pivot.loc[top_communities]

ax = pivot.plot(kind="bar", stacked=True, figsize=(12, 6))

ax.set_title("Subreddit composition of top Louvain communities")
ax.set_xlabel("Louvain community")
ax.set_ylabel("Share of users (%)")
ax.legend(title="Primary subreddit", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig("figures/louvain_subreddit_composition.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved figures/louvain_subreddit_composition.png")

## Cross-Subreddit Interaction and Bridge Users

This section measures how much interaction occurs within versus across subreddits and identifies users who connect multiple subreddit communities.

In [ ]:
# how much of the interaction is actually within-subreddit vs across?
edges_df['src_main_sub']  = edges_df['src'].map(user_main_sub)
edges_df['dst_main_sub']  = edges_df['dst'].map(user_main_sub)
edges_df['cross_sub'] = edges_df['src_main_sub'] != edges_df['dst_main_sub']

print(f"Total edges: {len(edges_df):,}")
print(f"Within-subreddit:  {(~edges_df['cross_sub']).sum():,} "
      f"({(~edges_df['cross_sub']).mean():.1%})")
print(f"Cross-subreddit:   {edges_df['cross_sub'].sum():,} "
      f"({edges_df['cross_sub'].mean():.1%})")

# who are the cross-subreddit bridge users?
cross_edges = edges_df[edges_df['cross_sub']]
bridges = pd.concat([cross_edges['src'], cross_edges['dst']]).value_counts().head(15)
print("\nTop 15 cross-subreddit bridge users:")
print(bridges.to_string())

# are your high-betweenness users from before the same as your cross-sub bridges?
print("\nOverlap with top-15 betweenness users:")
top_betweenness = set(cent_df.nlargest(15, 'betweenness')['user'])
print(set(bridges.head(15).index) & top_betweenness)

## Final Cross-Subreddit Network Figure Export

This cell regenerates the cross-subreddit interaction network figure from the final edge list and cached Reddit datasets. It ensures the saved PNG matches the final report’s network analysis rather than the removed exploratory pipeline.

In [ ]:
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

edges_df = pd.read_csv("data/edges.csv")
comments_df = pd.read_csv("data/reddit_comments.csv")
posts_df = pd.read_csv("data/reddit_posts.csv")

activity = pd.concat(
    [
        comments_df[["author", "subreddit"]],
        posts_df[["author", "subreddit"]]
    ],
    ignore_index=True
)

activity = activity[
    ~activity["author"].isin(["[deleted]", "AutoModerator", ""])
].dropna(subset=["author", "subreddit"])

user_main_sub = (
    activity
    .groupby(["author", "subreddit"])
    .size()
    .reset_index(name="n")
    .sort_values(["author", "n"], ascending=[True, False])
    .drop_duplicates("author")
)

user_to_sub = dict(zip(user_main_sub["author"], user_main_sub["subreddit"]))

edges_df["src_main_sub"] = edges_df["src"].map(user_to_sub)
edges_df["dst_main_sub"] = edges_df["dst"].map(user_to_sub)

sub_edges = (
    edges_df
    .dropna(subset=["src_main_sub", "dst_main_sub"])
    .groupby(["src_main_sub", "dst_main_sub"])
    .size()
    .reset_index(name="weight")
)

SG = nx.DiGraph()

for _, row in sub_edges.iterrows():
    SG.add_edge(row["src_main_sub"], row["dst_main_sub"], weight=row["weight"])

plt.figure(figsize=(10, 8))

pos = nx.spring_layout(SG, seed=42, k=1.8)
edge_weights = [SG[u][v]["weight"] / 150 for u, v in SG.edges()]

nx.draw_networkx_nodes(SG, pos, node_size=4000)
nx.draw_networkx_labels(SG, pos, font_size=10)
nx.draw_networkx_edges(
    SG,
    pos,
    width=edge_weights,
    arrows=True,
    arrowsize=16,
    alpha=0.6
)

plt.title("Cross-subreddit interaction network")
plt.axis("off")
plt.tight_layout()
plt.savefig("figures/cross_subreddit_network.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved figures/cross_subreddit_network.png")
print(f"Edges used: {len(edges_df):,}")

## Sentiment Analysis using VADER

This section applies VADER sentiment analysis to compare the affective tone of tariff-related discussion across subreddits and detected communities.

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# load cached files so this cell works even after restarting the kernel
comments_df = pd.read_csv("data/reddit_comments.csv")
cent_df = pd.read_csv("data/centrality.csv")

vader = SentimentIntensityAnalyzer()

def score_text(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return vader.polarity_scores(text)["compound"]

print("Scoring comments... (~1-2 min)")
comments_df["sentiment"] = comments_df["body"].apply(score_text)

print("\nMean sentiment per subreddit:")
print(comments_df.groupby("subreddit")["sentiment"].agg(["mean", "median", "count"]).round(3))

# sentiment per user
user_sentiment = (
    comments_df
    .groupby("author")
    .agg(
        mean_sentiment=("sentiment", "mean"),
        n_comments=("sentiment", "count")
    )
)

user_sentiment = user_sentiment[user_sentiment["n_comments"] >= 3]

# to avoid duplicate columns if being rerun
cent_df = cent_df.drop(
    columns=[c for c in ["mean_sentiment", "n_comments"] if c in cent_df.columns],
    errors="ignore"
)

# merge user sentiment into centrality table
cent_df = cent_df.merge(
    user_sentiment,
    left_on="user",
    right_index=True,
    how="left"
)

print("\nSentiment distribution per community (top 5):")
for cid in range(5):
    sub = cent_df[(cent_df["community"] == cid) & (cent_df["mean_sentiment"].notna())]
    if len(sub):
        print(
            f"Community {cid}: mean={sub['mean_sentiment'].mean():.3f}, "
            f"median={sub['mean_sentiment'].median():.3f}, "
            f"n={len(sub)}"
        )

cent_df.to_csv("data/centrality.csv", index=False)
comments_df.to_csv("data/reddit_comments.csv", index=False)

print("\nSaved updated sentiment columns to data/centrality.csv and data/reddit_comments.csv")

## Archived Stance Classification Code

This cell preserves the original BART-MNLI zero-shot stance classification code. It is kept for transparency but should not be rerun during marking because the final analysis uses cached stance outputs and manual validation results.

In [ ]:
'''
# pip install transformers torch
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

# Stance labels — the hypothesis template matters
candidate_labels = [
    "supports Trump's tariffs",
    "opposes Trump's tariffs",
    "neutral about tariffs",
]
hypothesis_template = "This comment {}."

# Strategy: drop very short / non-substantive comments before classifying
# (saves time AND improves accuracy — "lol" has no stance)
def is_substantive(text):
    return isinstance(text, str) and len(text.strip().split()) >= 8

classifiable = comments_df[comments_df['body'].apply(is_substantive)].copy()
print(f"{len(classifiable):,} substantive comments available "
      f"(skipped {len(comments_df) - len(classifiable):,} short ones)")

classifiable = classifiable.sample(n=3000, random_state=42).copy()
print(f"Sampling {len(classifiable):,} for classification")

# Batch through them — adjust batch_size based on your RAM/GPU
import time
results = []
batch_size = 16
texts = classifiable['body'].tolist()
# Truncate very long comments — BART has a 1024 token limit
texts = [t[:2000] for t in texts]

start = time.time()
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    out = classifier(
        batch,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
    )
    # pipeline returns a list when given a list
    if isinstance(out, dict):
        out = [out]
    for r in out:
        # top label and its score
        top_idx = 0  # already sorted by score desc
        results.append({
            'top_label': r['labels'][top_idx],
            'top_score': r['scores'][top_idx],
            'pro_score':     r['scores'][r['labels'].index("supports Trump's tariffs")],
            'anti_score':    r['scores'][r['labels'].index("opposes Trump's tariffs")],
            'neutral_score': r['scores'][r['labels'].index("neutral about tariffs")],
        })
    if (i // batch_size) % 10 == 0:
        elapsed = time.time() - start
        rate = (i + batch_size) / elapsed if elapsed > 0 else 0
        eta = (len(texts) - i) / rate if rate > 0 else 0
        print(f"  {i+batch_size:,}/{len(texts):,}  ({rate:.1f}/s, ETA {eta/60:.1f} min)")

stance_df = pd.DataFrame(results, index=classifiable.index)
classifiable = pd.concat([classifiable, stance_df], axis=1)

# Map back to single 'stance' label: pro / anti / neutral
def to_stance(row):
    if row['top_score'] < 0.45:        # low-confidence → neutral
        return 'neutral'
    if 'supports' in row['top_label']:
        return 'pro'
    if 'opposes' in row['top_label']:
        return 'anti'
    return 'neutral'

classifiable['stance'] = classifiable.apply(to_stance, axis=1)

# Merge back into comments_df
comments_df = comments_df.merge(
    classifiable[['stance', 'pro_score', 'anti_score', 'neutral_score']],
    left_index=True, right_index=True, how='left'
)
comments_df['stance'] = comments_df['stance'].fillna('neutral')  # short comments default to neutral

comments_df.to_csv('data/reddit_comments.csv', index=False)

# How does stance distribute?
print("\nOverall stance distribution:")
print(comments_df['stance'].value_counts(normalize=True).round(3))

print("\nStance by subreddit:")
print(pd.crosstab(comments_df['subreddit'], comments_df['stance'], normalize='index').round(3))
'''

## Stance Output Loading and Validation

This section reloads the cached stance-classification outputs and checks the available stance-related columns. These cached outputs support the manual validation and methodological pivot from stance analysis to frame analysis.

In [ ]:
comments_df = pd.read_csv("data/reddit_comments.csv")
comments_df.columns

In [ ]:
# sanity check, examples of each stance
for stance in ['pro', 'anti', 'neutral']:
    print(f"\n=== Examples classified as {stance.upper()} ===")

    subset = comments_df[comments_df['stance'] == stance]

    if len(subset) == 0:
        print("No examples found.")
        continue

    examples = subset.sample(min(5, len(subset)), random_state=1)

    for _, row in examples.iterrows():
        confidence = max(
            row.get('pro_score', 0),
            row.get('anti_score', 0),
            row.get('neutral_score', 0)
        )

        print(f"\n[confidence {confidence:.2f}] r/{row['subreddit']}")
        print(f"  {str(row['body'])[:300]}")

In [ ]:
required = ["stance", "pro_score", "anti_score", "neutral_score"]

if all(col in comments_df.columns for col in required):
    print("Stance labels already loaded from reddit_comments.csv")
    print(comments_df["stance"].value_counts(normalize=True).round(3))
else:
    print("Stance columns missing — run the stance classification cell.")

## Topic Modelling and Frame Analysis

This section applies LDA topic modelling to identify dominant discourse frames in the Reddit comments. The resulting topics are used to analyse how tariff discussion was framed across different subreddit communities.

In [ ]:
import re

# drops link-heavy comments
def has_link(text):
    return bool(re.search(r'http\S+|www\.\S+', str(text)))

clean = comments_df[
    (comments_df['body'].str.split().str.len() >= 15) &
    (~comments_df['body'].apply(has_link))
].copy()

extra_stops = ['just', 'like', 'don', 'know', 'think', 'going',
               'll', 've', 'right', 'people', 'need', 'years',
               'time', 'day', 'make', 'really', 'said', 'say',
               'thing', 'way', 'good', 'lot', 'gonna']

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vec = CountVectorizer(
    max_features=2000,
    stop_words=list(set(CountVectorizer(stop_words='english').get_stop_words()).union(extra_stops)),
    min_df=15,
    max_df=0.4, # drop common/familiar words
    ngram_range=(1, 2),
)
X = vec.fit_transform(clean['body'].fillna(''))
vocab = vec.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=6, random_state=42,
                                learning_method='online', n_jobs=-1)
doc_topics = lda.fit_transform(X)

# assign each comment its dominant topic
clean['topic'] = doc_topics.argmax(axis=1)
clean['topic_confidence'] = doc_topics.max(axis=1)

# label each topic with its top words
topic_labels = {}
for i, t in enumerate(lda.components_):
    top = t.argsort()[-8:][::-1]
    topic_labels[i] = ' · '.join(vocab[j] for j in top)
    print(f"Topic {i}: {topic_labels[i]}")

print(f"\nClean dataset size: {len(clean):,}")
print(f"\nTopic distribution:")
print(clean['topic'].value_counts().sort_index())

## Topic Distribution Across Subreddits

This section compares topic assignments across subreddits to show whether different communities framed the tariff discussion differently.

In [ ]:
print("Topic distribution by subreddit (row %):")
crosstab = pd.crosstab(clean['subreddit'], clean['topic'], normalize='index').round(3)
crosstab.columns = [f"T{i}: {topic_labels[i].split(chr(183))[0].strip()}" for i in crosstab.columns]
print(crosstab)

In [ ]:
import os
os.makedirs('figures', exist_ok=True)

## Frame Heatmap Visualisation

This section generates the topic-by-subreddit heatmap used in the report to visualise how discourse frames vary across Australian Reddit communities.

In [ ]:
import os
os.makedirs('figures', exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns

topic_names = {
    0: 'Consumer\neconomics',
    1: 'Industry\nimpacts',
    2: 'Global\ntrade',
    3: 'Australian\npolitics',
    4: 'Investment\n& markets',
    5: 'China &\nsecurity',
}

ct = pd.crosstab(clean['subreddit'], clean['topic'], normalize='index') * 100
ct.columns = [topic_names[i] for i in ct.columns]

subreddit_order = ['ASX', 'ASX_Bets', 'AusFinance', 'AusEcon', 'AustralianPolitics', 'australia']
ct = ct.reindex(subreddit_order)
ct.index = ['r/' + s for s in ct.index]

fig, ax = plt.subplots(figsize=(11, 5.5))

annot = ct.round(0).astype(int).astype(str) + '%'

sns.heatmap(
    ct,
    annot=annot.values,
    fmt='',
    cmap='YlOrBr',
    cbar_kws={'label': 'Share of subreddit comments (%)', 'pad': 0.02},
    linewidths=1,
    linecolor='white',
    annot_kws={'fontsize': 11, 'fontweight': 'bold'},
    vmin=0, vmax=55,
    ax=ax,
)

for text, val in zip(ax.texts, ct.values.flatten()):
    text.set_color('white' if val > 27 else '#3a2a14')

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('Frame distribution across Australian Reddit subreddits',
             pad=14, fontsize=14, fontweight='semibold')
plt.xticks(rotation=0, fontsize=10)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig('figures/topic_subreddit_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nDominant frame per subreddit:")
for sub in subreddit_order:
    top_topic = ct.loc['r/' + sub].idxmax().replace('\n', ' ')
    top_pct = ct.loc['r/' + sub].max()
    print(f"  r/{sub:20s} → {top_topic:25s} ({top_pct:.0f}%)")

## Qualitative Topic Checks

This section samples comments from selected topic-subreddit combinations to check whether the LDA topic labels are meaningful and interpretable in the actual Reddit text.

In [ ]:
ap_t0 = clean[(clean['subreddit'] == 'AustralianPolitics') & (clean['topic'] == 0)]
print(f"r/AustralianPolitics comments classified as Topic 0 ({len(ap_t0)} total)")
print(f"Showing 10 random examples:\n")
for _, row in ap_t0.sample(n=10, random_state=42).iterrows():
    body = row['body'][:300].replace('\n', ' ')
    print(f"• {body}\n")

In [ ]:
au_t4 = clean[(clean['subreddit'] == 'australia') & (clean['topic'] == 4)]
print(f"r/australia comments classified as Topic 4 ({len(au_t4)} total)")
print(f"Showing 10 random examples:\n")
for _, row in au_t4.sample(n=10, random_state=42).iterrows():
    body = row['body'][:300].replace('\n', ' ')
    print(f"• {body}\n")

## Temporal Diffusion Analysis

This section visualises daily tariff-related comment volume across the six Australian subreddits and examines how discussion changed around the 2 April 2025 “Liberation Day” tariff announcement.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# daily comment volume
comments_df['date'] = pd.to_datetime(comments_df['created_utc'], unit='s', utc=True)
daily = comments_df.groupby([comments_df['date'].dt.date, 'subreddit']).size().unstack(fill_value=0)

# Liberation Day
LIBERATION_DAY = datetime(2025, 4, 2).date()

fig, ax = plt.subplots(figsize=(13, 6))

subreddit_order = ['ASX', 'ASX_Bets', 'AusFinance', 'AusEcon', 'AustralianPolitics', 'australia']
colors = ['#5b8a72', '#c89b4a', '#b85745', '#7a6b8a', '#6b8a52', '#8a7b6b']
ax.stackplot(daily.index,
             [daily[s].values for s in subreddit_order if s in daily.columns],
             labels=['r/' + s for s in subreddit_order if s in daily.columns],
             colors=colors, alpha=0.85, edgecolor='white', linewidth=0.3)

# marks Liberation Day
ax.axvline(LIBERATION_DAY, color='#c4452f', linestyle='--', linewidth=1.5, alpha=0.7, zorder=10)
ax.annotate('Liberation Day\n(2 April 2025)',
            xy=(LIBERATION_DAY, ax.get_ylim()[1] * 0.85),
            xytext=(10, 0), textcoords='offset points',
            fontsize=10, color='#c4452f', fontweight='semibold',
            verticalalignment='top')

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=0)
ax.set_ylabel('Daily comment volume')
ax.set_xlabel('')
ax.set_title('Comment volume across Australian Reddit, 2025', fontsize=13, pad=12)
ax.legend(loc='upper right', fontsize=9, framealpha=0.95)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('figures/diffusion_timeline.png', dpi=200, bbox_inches='tight')
plt.show()

# summary
print("\nComment volume — week-by-week around Liberation Day:")
liberation_dt = pd.Timestamp('2025-04-02', tz='UTC')
windows = {
    'Two weeks before':  (liberation_dt - pd.Timedelta(days=14), liberation_dt - pd.Timedelta(days=1)),
    'Liberation Day week': (liberation_dt, liberation_dt + pd.Timedelta(days=6)),
    'Week after':        (liberation_dt + pd.Timedelta(days=7), liberation_dt + pd.Timedelta(days=13)),
    'Two weeks after':   (liberation_dt + pd.Timedelta(days=14), liberation_dt + pd.Timedelta(days=20)),
}
for label, (s, e) in windows.items():
    n = ((comments_df['date'] >= s) & (comments_df['date'] <= e)).sum()
    print(f"  {label:25s} {n:>5,} comments")